# Estudo de Caso 9.2 — Condensador de Vapor para UTE a Gás (100 MW)

**Capítulo Associado:** Capítulo 9 — Trocadores de Calor  
**Nível:** Graduação/Pós-Graduação  
**Tempo estimado:** 90 minutos  
**Autor:** Jader Lugon Junior

---

## 📋 Resumo

Neste estudo de caso, dimensionamos o **condensador de vapor** de uma usina termelétrica (UTE) a gás natural com potência elétrica nominal de **100 MW** e eficiência térmica de **40%**. O condensador utiliza água captada de um rio como fluido de resfriamento para liquefazer o vapor de exaustão da turbina, fechando o ciclo Rankine.

**Objetivos:**
1. Realizar o balanço energético da UTE
2. Calcular a vazão de água de resfriamento necessária
3. Determinar a LMTD e a área de troca térmica
4. Configurar fisicamente o condensador (número de tubos, dimensões)
5. Analisar o impacto da variação sazonal da temperatura do rio
6. Avaliar o impacto ambiental do lançamento de água aquecida

---

## 🎯 Objetivos de Aprendizagem

- Aplicar balanço de energia em ciclo termodinâmico real
- Dimensionar trocador de calor com mudança de fase (condensação)
- Calcular LMTD para fluido em temperatura constante
- Configurar fisicamente trocadores casco-tubo de grande porte
- Analisar sensibilidade a variações operacionais (temperatura do rio)
- Avaliar impacto ambiental de efluentes térmicos

---

## 🔧 Requisitos

- Bibliotecas: `numpy`, `matplotlib`
- Conhecimento prévio: LMTD, método ε-NTU, balanço de energia (Cap. 9)
- Conceitos de ciclo Rankine e propriedades do vapor d'água

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('Ambiente configurado com sucesso!')

---

## 🏭 Seção 1: Contexto Industrial

Uma **usina termelétrica (UTE)** a gás natural opera com um ciclo Rankine modificado. O gás natural é queimado em uma câmara de combustão, gerando gases quentes que aquecem água em uma caldeira, produzindo vapor em alta pressão. Esse vapor expande-se através de uma turbina, gerando eletricidade. Após passar pela turbina, o vapor precisa ser **condensado** (liquefeito) para retornar à caldeira, fechando o ciclo.

O **condensador** é o trocador de calor responsável por essa condensação. Ele opera com:
- **Fluido quente:** vapor d'água em baixa pressão (condensando a temperatura constante)
- **Fluido frio:** água captada de um rio (aquecendo sem mudança de fase)

A eficiência do condensador impacta diretamente a eficiência global da usina: quanto menor a temperatura de condensação, maior o trabalho útil extraído da turbina.

---

## ⚡ Seção 2: Balanço Energético da UTE

### Dados de Projeto

| Parâmetro | Valor | Unidade |
|-----------|-------|---------|
| Potência elétrica | $W_{el} = 100$ | MW |
| Eficiência térmica | $\eta = 40\%$ | adimensional |
| Temperatura de condensação | $T_{vap} = 50$ | °C |
| Pressão de condensação | $p_{vap} \approx 12,3$ | kPa |
| Calor latente de vaporização | $h_{fg} = 2383$ | kJ/kg |
| Temperatura da água (entrada) | $T_{agua,e} = 22$ | °C |
| Temperatura da água (saída) | $T_{agua,s} = 32$ | °C |
| Calor específico da água | $c_{p,agua} = 4180$ | J/(kg·K) |
| Densidade da água | $\rho_{agua} = 998$ | kg/m³ |
| Coeficiente global | $U = 1800$ | W/(m²·K) |

### Passo 1: Calor Fornecido pela Caldeira

Pela definição de eficiência térmica:

$$\eta = \frac{W_{el}}{Q_{caldeira}} \quad \Rightarrow \quad Q_{caldeira} = \frac{W_{el}}{\eta}$$

Substituindo:

$$Q_{caldeira} = \frac{100}{0,40} = 250 \text{ MW}$$

### Passo 2: Calor Rejeitado no Condensador

Pelo balanço de energia global da UTE:

$$Q_{cond} = Q_{caldeira} - W_{el} = 250 - 100 = 150 \text{ MW}$$

**Interpretação:** A cada 250 MW de calor fornecido pela queima de gás, apenas 100 MW são convertidos em eletricidade. Os restantes **150 MW são rejeitados no condensador** — uma quantidade expressiva que justifica a importância deste equipamento para a eficiência global da usina.

In [ ]:
# ============================================================
# BALANÇO ENERGÉTICO DA UTE
# ============================================================

# Dados da UTE
W_el = 100  # MW
eta = 0.40  # eficiência térmica

# Cálculos
Q_caldeira = W_el / eta  # MW
Q_cond = Q_caldeira - W_el  # MW

print('=' * 60)
print('BALANÇO ENERGÉTICO DA UTE')
print('=' * 60)
print(f'\nPotência elétrica: W_el = {W_el} MW')
print(f'Eficiência térmica: η = {eta*100:.0f}%')
print(f'\nCalor fornecido pela caldeira: Q_caldeira = {Q_caldeira:.0f} MW')
print(f'Calor rejeitado no condensador: Q_cond = {Q_cond:.0f} MW')
print(f'\nFração rejeitada: {Q_cond/Q_caldeira*100:.0f}% do calor total')

# Gráfico de Sankey simplificado
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(['UTE'], [Q_caldeira], color='red', alpha=0.6, label='Calor da caldeira')
ax.barh(['UTE'], [W_el], color='green', alpha=0.8, label='Eletricidade útil')
ax.barh(['UTE'], [Q_cond], left=[W_el], color='blue', alpha=0.6, label='Calor rejeitado')

ax.set_xlabel('Potência [MW]')
ax.set_title('Balanço Energético da UTE (100 MW)', fontweight='bold')
ax.legend(loc='upper right')
ax.set_xlim(0, 300)

# Anotações
ax.text(W_el/2, 0, f'{W_el} MW\n(40%)', ha='center', va='center', fontweight='bold', color='white')
ax.text(W_el + Q_cond/2, 0, f'{Q_cond} MW\n(60%)', ha='center', va='center', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

---

## 💧 Seção 3: Vazão de Água de Resfriamento

### Condições da Água do Rio

A água do rio é captada e devolvida ao curso d'água após o resfriamento. Por questões ambientais e operacionais, a elevação de temperatura é limitada a **10°C** (dentro dos limites ambientais típicos).

### Passo 3: Vazão Mássica de Água

Pelo balanço de energia no lado frio:

$$Q_{cond} = \dot{m}_{agua} \cdot c_{p,agua} \cdot (T_{agua,s} - T_{agua,e})$$

Isolando a vazão mássica:

$$\dot{m}_{agua} = \frac{Q_{cond}}{c_{p,agua} \cdot (T_{agua,s} - T_{agua,e})}$$

Substituindo:

$$\dot{m}_{agua} = \frac{150.000}{4180 \times 10} \approx 3589 \text{ kg/s}$$

### Passo 4: Vazão Volumétrica

$$\dot{V}_{agua} = \frac{\dot{m}_{agua}}{\rho_{agua}} = \frac{3589}{998} \approx 3,6 \text{ m}^3\text{/s}$$

**Interpretação prática:** A vazão de 3,6 m³/s equivale a **3.600 litros por segundo** — um volume considerável que exige bombeamento potente e cuidados com o impacto ambiental no rio.

In [ ]:
# ============================================================
# VAZÃO DE ÁGUA DE RESFRIAMENTO
# ============================================================

# Dados
Q_cond_MW = 150  # MW
Q_cond = Q_cond_MW * 1e6  # W
cp_agua = 4180  # J/(kg·K)
rho_agua = 998  # kg/m³
T_agua_e = 22  # °C
T_agua_s = 32  # °C
delta_T = T_agua_s - T_agua_e  # °C

# Cálculos
m_agua = Q_cond / (cp_agua * delta_T)  # kg/s
V_agua = m_agua / rho_agua  # m³/s

print('=' * 60)
print('VAZÃO DE ÁGUA DE RESFRIAMENTO')
print('=' * 60)
print(f'\nTemperatura de entrada: T_agua,e = {T_agua_e}°C')
print(f'Temperatura de saída: T_agua,s = {T_agua_s}°C')
print(f'Elevação de temperatura: ΔT = {delta_T}°C')
print(f'\nVazão mássica: ṁ_agua = {m_agua:.0f} kg/s')
print(f'Vazão volumétrica: V̇_agua = {V_agua:.2f} m³/s')
print(f'Vazão em L/s: {V_agua*1000:.0f} L/s')

# Comparação com objetos do cotidiano
print(f'\n📊 Comparação:')
print(f'  • Equivale a {V_agua*1000/10:.0f} chuveiros ligados simultaneamente')
print(f'  • Enche uma piscina olímpica (2.500 m³) em {2500/V_agua/60:.1f} min')
print(f'  • Representa {V_agua/0.1*100:.0f}% da vazão de uma torneira industrial')

---

## 🌡️ Seção 4: Cálculo da LMTD

### Característica Especial: Mudança de Fase

Como o vapor condensa a **temperatura constante** ($T_{vap} = 50°C$), o perfil de temperatura é característico de um fluido em mudança de fase. Nesse caso, a LMTD é a mesma tanto para configurações em paralelo quanto em contracorrente.

### Passo 5: Diferenças de Temperatura nas Extremidades

$$\Delta T_1 = T_{vap} - T_{agua,e} = 50 - 22 = 28°C$$

$$\Delta T_2 = T_{vap} - T_{agua,s} = 50 - 32 = 18°C$$

### Passo 6: LMTD

$$\Delta T_{lm} = \frac{\Delta T_1 - \Delta T_2}{\ln(\Delta T_1 / \Delta T_2)} = \frac{28 - 18}{\ln(28/18)} = \frac{10}{0,442} \approx 22,6°C$$

In [ ]:
# ============================================================
# CÁLCULO DA LMTD
# ============================================================

# Dados
T_vap = 50  # °C (temperatura de condensação)

# Diferenças de temperatura
dT1 = T_vap - T_agua_e
dT2 = T_vap - T_agua_s

# LMTD
LMTD = (dT1 - dT2) / np.log(dT1 / dT2)

# Média aritmética (para comparação)
LMTD_arith = (dT1 + dT2) / 2

# Erro da média aritmética
erro_arith = abs(LMTD_arith - LMTD) / LMTD * 100

print('=' * 60)
print('CÁLCULO DA LMTD')
print('=' * 60)
print(f'\nTemperatura do vapor: T_vap = {T_vap}°C (constante)')
print(f'\nDiferenças de temperatura:')
print(f'  • ΔT₁ = T_vap - T_agua,e = {dT1}°C')
print(f'  • ΔT₂ = T_vap - T_agua,s = {dT2}°C')
print(f'  • Razão ΔT₁/ΔT₂ = {dT1/dT2:.2f}')
print(f'\nLMTD = {LMTD:.1f}°C')
print(f'Média aritmética = {LMTD_arith:.1f}°C')
print(f'Erro da média aritmética = {erro_arith:.1f}%')

# Gráfico do perfil de temperatura
fig, ax = plt.subplots(figsize=(10, 5))

# Eixo x: posição no trocador (0 a 1)
x = np.linspace(0, 1, 100)

# Vapor (temperatura constante)
T_vap_profile = np.full_like(x, T_vap)

# Água (perfil exponencial decrescente em contracorrente)
# Em contracorrente: água entra fria (x=1) e sai quente (x=0)
T_agua_profile = T_agua_e + (T_agua_s - T_agua_e) * (1 - x)

ax.plot(x, T_vap_profile, 'r-', linewidth=3, label='Vapor (condensando)')
ax.plot(x, T_agua_profile, 'b-', linewidth=3, label='Água (aquecendo)')

# Preencher área entre as curvas
ax.fill_between(x, T_agua_profile, T_vap_profile, alpha=0.2, color='orange',
                label=f'LMTD = {LMTD:.1f}°C')

# Anotações
ax.annotate(f'ΔT₁ = {dT1}°C', xy=(0, T_vap), xytext=(0.1, T_vap - 3),
            fontsize=10, fontweight='bold', color='red')
ax.annotate(f'ΔT₂ = {dT2}°C', xy=(1, T_vap), xytext=(0.8, T_vap - 3),
            fontsize=10, fontweight='bold', color='red')

ax.set_xlabel('Posição no Trocador')
ax.set_ylabel('Temperatura [°C]')
ax.set_title('Perfil de Temperatura no Condensador', fontweight='bold')
ax.legend(loc='best')
ax.set_xlim(0, 1)
ax.set_ylim(15, 55)
ax.set_xticks([0, 0.5, 1])
ax.set_xticklabels(['Entrada vapor\nSaída água', 'Meio', 'Saída vapor\nEntrada água'])

plt.tight_layout()
plt.show()

---

## 🔧 Seção 5: Dimensionamento da Área de Troca

### Tipo de Trocador e Coeficiente Global

Para condensação de vapor com água como fluido de resfriamento, o tipo mais comum é o **casco e tubos** (shell and tube), com:
- Vapor condensando no casco (lado externo dos tubos)
- Água circulando nos tubos (alta velocidade para evitar incrustação)
- Tubos de cobre-níquel (boa condutividade e resistência à corrosão)
- Múltiplas passadas nos tubos para aumentar a velocidade da água

O coeficiente global de transferência de calor típico para esta aplicação é:

$$U = 1800 \text{ W/(m}^2\cdot\text{K)}$$

**Por que $U$ é tão alto?** A condensação de vapor é um processo extremamente eficiente, com coeficientes de película da ordem de $5000$ a $15000$ W/(m²·K). A água nos tubos também tem coeficiente alto ($3000$ a $6000$ W/(m²·K)). A resistência dominante é a incrustação nos tubos e a condução através da parede metálica.

### Passo 7: Área de Troca Necessária

Aplicando a equação fundamental de dimensionamento:

$$A = \frac{Q_{cond}}{U \cdot \Delta T_{lm}} = \frac{150.000.000}{1800 \times 22,6} \approx 3689 \text{ m}^2$$

Arredondando para um valor comercial:

$$\boxed{A \approx 3700 \text{ m}^2}$$

In [ ]:
# ============================================================
# DIMENSIONAMENTO DA ÁREA DE TROCA
# ============================================================

# Dados
U = 1800  # W/(m²·K)

# Cálculo da área
A_req = Q_cond / (U * LMTD)  # m²
A_comercial = np.ceil(A_req / 100) * 100  # arredondar para múltiplo de 100

print('=' * 60)
print('DIMENSIONAMENTO DA ÁREA DE TROCA')
print('=' * 60)
print(f'\nCoeficiente global: U = {U} W/(m²·K)')
print(f'LMTD: ΔT_lm = {LMTD:.1f}°C')
print(f'\nÁrea requerida: A = {A_req:.0f} m²')
print(f'Área comercial: A ≈ {A_comercial:.0f} m²')

# Comparação visual
print(f'\n📊 Para visualizar {A_comercial:.0f} m²:')
print(f'  • Equivale a {A_comercial/100:.0f} apartamentos de 100 m²')
print(f'  • Ou um campo de futebol ({A_comercial/7140*100:.1f}% de um campo)')
print(f'  • Ou {A_comercial/1.2:.0f} painéis solares padrão')

---

## 🏗️ Seção 6: Configuração Física do Condensador

Para uma área de 3700 m², um condensador típico teria:

| Parâmetro | Valor |
|-----------|-------|
| Diâmetro do casco | $\approx 3$ m |
| Comprimento dos tubos | $\approx 6$ m |
| Número de tubos | $\approx 3500$ |
| Diâmetro dos tubos | $\varnothing$ 20 mm $\times$ 1 mm |
| Número de passadas | 2 ou 4 |
| Material dos tubos | Cobre-níquel (90/10 ou 70/30) |

### Passo 8: Verificação do Número de Tubos

A área externa de um tubo é:

$$A_{tubo} = \pi \cdot D_e \cdot L = \pi \times 0,020 \times 6 = 0,377 \text{ m}^2$$

O número de tubos necessários é:

$$N_{tubos} = \frac{A}{A_{tubo}} = \frac{3700}{0,377} \approx 9814 \text{ tubos}$$

**Nota:** O valor calculado (9814 tubos) é maior que a estimativa inicial (3500 tubos) porque a estimativa considerou uma configuração mais compacta. Na prática, o projeto final depende do arranjo dos tubos (triangular ou quadrado), passo e número de passadas.

In [ ]:
# ============================================================
# CONFIGURAÇÃO FÍSICA DO CONDENSADOR
# ============================================================

# Dados geométricos
D_e = 0.020  # m (diâmetro externo)
D_i = 0.018  # m (diâmetro interno)
L_tubo = 6.0  # m (comprimento)

# Área por tubo
A_tubo = np.pi * D_e * L_tubo

# Número de tubos
N_tubos = A_comercial / A_tubo

print('=' * 60)
print('CONFIGURAÇÃO FÍSICA DO CONDENSADOR')
print('=' * 60)
print(f'\nDiâmetro externo: D_e = {D_e*1000:.0f} mm')
print(f'Diâmetro interno: D_i = {D_i*1000:.0f} mm')
print(f'Comprimento: L = {L_tubo:.0f} m')
print(f'\nÁrea por tubo: A_tubo = {A_tubo:.3f} m²')
print(f'Número de tubos: N = {N_tubos:.0f} tubos')

# Área interna total
A_int_total = N_tubos * np.pi * D_i * L_tubo
print(f'\nÁrea interna total: A_int = {A_int_total:.0f} m²')
print(f'Razão A_ext/A_int = {A_comercial/A_int_total:.2f}')

# Visualização esquemática
fig, ax = plt.subplots(figsize=(10, 6))

# Casco (círculo externo)
casco = plt.Circle((0.5, 0.5), 0.4, fill=False, linewidth=3, color='black')
ax.add_patch(casco)

# Tubos (círculos internos)
np.random.seed(42)
for _ in range(200):
    # Gerar pontos aleatórios dentro do casco
    r = 0.35 * np.sqrt(np.random.random())
    theta = 2 * np.pi * np.random.random()
    x = 0.5 + r * np.cos(theta)
    y = 0.5 + r * np.sin(theta)
    tubo = plt.Circle((x, y), 0.008, fill=True, color='blue', alpha=0.6)
    ax.add_patch(tubo)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Vista Transversal do Condensador (esquemática)', fontweight='bold')

# Anotações
ax.text(0.5, 0.95, f'Diâmetro do casco ≈ 3 m', ha='center', fontsize=10)
ax.text(0.5, -0.05, f'~{N_tubos:.0f} tubos de cobre-níquel', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

---

## 📊 Seção 7: Verificação da Velocidade da Água

### Passo 9: Área de Escoamento nos Tubos

A área interna de um tubo é:

$$A_{int,tubo} = \frac{\pi D_i^2}{4} = \frac{\pi (0,018)^2}{4} = 2,54 \times 10^{-4} \text{ m}^2$$

A área total de escoamento é:

$$A_{int,total} = N_{tubos} \cdot A_{int,tubo} = 9814 \times 2,54 \times 10^{-4} = 2,49 \text{ m}^2$$

### Passo 10: Velocidade da Água

$$v_{agua} = \frac{\dot{V}_{agua}}{A_{int,total}} = \frac{3,6}{2,49} \approx 1,44 \text{ m/s}$$

**Verificação:** A velocidade de 1,44 m/s está **ligeiramente abaixo** da faixa recomendada (1,5 a 2,5 m/s) para evitar incrustação e erosão. Isso indica que o projeto pode exigir ajuste no número de passadas ou no diâmetro dos tubos.

In [ ]:
# ============================================================
# VERIFICAÇÃO DA VELOCIDADE DA ÁGUA
# ============================================================

# Área interna por tubo
A_int_tubo = np.pi * D_i**2 / 4

# Área total de escoamento
A_int_total = N_tubos * A_int_tubo

# Velocidade da água
v_agua = V_agua / A_int_total

print('=' * 60)
print('VERIFICAÇÃO DA VELOCIDADE DA ÁGUA')
print('=' * 60)
print(f'\nÁrea interna por tubo: A_int,tubo = {A_int_tubo:.2e} m²')
print(f'Área total de escoamento: A_int,total = {A_int_total:.2f} m²')
print(f'\nVelocidade da água: v = {v_agua:.2f} m/s')

# Verificação da faixa recomendada
v_min, v_max = 1.5, 2.5
if v_min <= v_agua <= v_max:
    print(f'\n✅ Velocidade dentro da faixa recomendada ({v_min}-{v_max} m/s)')
elif v_agua < v_min:
    print(f'\n⚠️ Velocidade ABAIXO da faixa recomendada ({v_min}-{v_max} m/s)')
    print(f'   Risco de incrustação e biofilme')
else:
    print(f'\n⚠️ Velocidade ACIMA da faixa recomendada ({v_min}-{v_max} m/s)')
    print(f'   Risco de erosão e perda de carga excessiva')

# Número de Reynolds
nu_agua = 1.0e-6  # m²/s (viscosidade cinemática a 20°C)
Re = v_agua * D_i / nu_agua

print(f'\nNúmero de Reynolds: Re = {Re:.0f}')
if Re > 4000:
    print(f'   Regime: TURBULENTO (Re > 4000)')
elif Re < 2300:
    print(f'   Regime: LAMINAR (Re < 2300)')
else:
    print(f'   Regime: TRANSIÇÃO (2300 < Re < 4000)')

---

## 📋 Seção 8: Resumo dos Resultados

| Grandeza | Valor |
|----------|-------|
| Carga térmica $Q_{cond}$ | 150 MW |
| Temperatura de condensação | 50°C |
| Vazão de vapor | 63 kg/s |
| Vazão de água | 3,6 m³/s |
| LMTD | 22,6°C |
| Coeficiente global $U$ | 1800 W/(m²·K) |
| Área de troca necessária | 3700 m² |
| Número de tubos | ~9800 |
| Velocidade da água | 1,44 m/s |

---

## 🔍 Seção 9: Análise de Sensibilidade — Efeito da Temperatura do Rio

### Cenário: Verão com Estiagem

Se a temperatura da água do rio subir para **28°C** no verão (devido a uma estiagem), qual seria a nova área de troca necessária para manter a mesma carga térmica?

### Passo 11: Nova LMTD

Mantendo $T_{agua,s} = 38°C$ (elevação de 10°C):

$$\Delta T_1 = T_{vap} - T_{agua,e} = 50 - 28 = 22°C$$

$$\Delta T_2 = T_{vap} - T_{agua,s} = 50 - 38 = 12°C$$

$$\Delta T_{lm,novo} = \frac{22 - 12}{\ln(22/12)} = \frac{10}{0,606} \approx 16,5°C$$

### Passo 12: Nova Área Necessária

$$A_{novo} = \frac{Q_{cond}}{U \cdot \Delta T_{lm,novo}} = \frac{150.000.000}{1800 \times 16,5} \approx 5050 \text{ m}^2$$

**Conclusão:** A área necessária aumenta de 3700 m² para **5050 m²** (aumento de 36%). Isso ilustra a sensibilidade do condensador às condições ambientais.

**Implicações para a eficiência da usina:**
- Com área fixa de 3700 m², a usina não conseguirá condensar todo o vapor a 50°C
- A temperatura de condensação aumentará, reduzindo a eficiência do ciclo Rankine
- A potência elétrica gerada cairá, ou será necessário reduzir a vazão de vapor

In [ ]:
# ============================================================
# ANÁLISE DE SENSIBILIDADE — TEMPERATURA DO RIO
# ============================================================

# Cenário de verão
T_agua_e_verao = 28  # °C
T_agua_s_verao = T_agua_e_verao + 10  # °C (mantendo ΔT = 10°C)

# Nova LMTD
dT1_verao = T_vap - T_agua_e_verao
dT2_verao = T_vap - T_agua_s_verao
LMTD_verao = (dT1_verao - dT2_verao) / np.log(dT1_verao / dT2_verao)

# Nova área
A_verao = Q_cond / (U * LMTD_verao)

# Aumento percentual
aumento_area = (A_verao - A_comercial) / A_comercial * 100

print('=' * 60)
print('ANÁLISE DE SENSIBILIDADE — VERÃO')
print('=' * 60)
print(f'\nTemperatura do rio (verão): T_agua,e = {T_agua_e_verao}°C')
print(f'Temperatura de saída: T_agua,s = {T_agua_s_verao}°C')
print(f'\nNova LMTD: ΔT_lm = {LMTD_verao:.1f}°C')
print(f'Nova área necessária: A = {A_verao:.0f} m²')
print(f'Área instalada: A = {A_comercial:.0f} m²')
print(f'\nAumento de área: {aumento_area:.0f}%')

# Gráfico de sensibilidade
T_agua_range = np.linspace(18, 32, 15)
A_range = []
LMTD_range = []

for T_agua_e_teste in T_agua_range:
    T_agua_s_teste = T_agua_e_teste + 10
    dT1_teste = T_vap - T_agua_e_teste
    dT2_teste = T_vap - T_agua_s_teste
    LMTD_teste = (dT1_teste - dT2_teste) / np.log(dT1_teste / dT2_teste)
    A_teste = Q_cond / (U * LMTD_teste)
    A_range.append(A_teste)
    LMTD_range.append(LMTD_teste)

A_range = np.array(A_range)
LMTD_range = np.array(LMTD_range)

fig, ax1 = plt.subplots(figsize=(10, 6))

color1 = 'tab:red'
ax1.set_xlabel('Temperatura do Rio [°C]', fontsize=12)
ax1.set_ylabel('Área Necessária [m²]', fontsize=12, color=color1)
ax1.plot(T_agua_range, A_range, color=color1, linewidth=2, marker='o')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(y=A_comercial, color='gray', linestyle='--', label=f'Área instalada ({A_comercial:.0f} m²)')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
color2 = 'tab:blue'
ax2.set_ylabel('LMTD [°C]', fontsize=12, color=color2)
ax2.plot(T_agua_range, LMTD_range, color=color2, linewidth=2, marker='s', linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

ax1.set_title('Efeito da Temperatura do Rio no Dimensionamento', fontweight='bold')
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n📊 Interpretação:')
print(f'  • Para T_agua,e = 18°C (inverno): A ≈ {A_range[0]:.0f} m²')
print(f'  • Para T_agua,e = 22°C (normal): A ≈ {A_range[4]:.0f} m²')
print(f'  • Para T_agua,e = 28°C (verão): A ≈ {A_verao:.0f} m²')
print(f'  • Para T_agua,e = 32°C (extremo): A ≈ {A_range[-1]:.0f} m²')

---

## 🌍 Seção 10: Impacto Ambiental

### Comparação com Vazão do Rio

A vazão de água necessária (3,6 m³/s) deve ser comparada com a vazão típica de um rio de médio porte:

| Tipo de Rio | Vazão Típica [m³/s] | Fração Captada |
|-------------|---------------------|----------------|
| Rio pequeno | 1--5 | 70--360% (inviável) |
| Rio médio | 50--100 | 3,6--7,2% |
| Rio grande | 500--1000 | 0,36--0,72% |

### Impactos do Lançamento de Água Aquecida

A água é devolvida ao rio com temperatura de 32°C (10°C acima da temperatura de entrada). Os impactos incluem:

1. **Poluição térmica:** O aumento de temperatura reduz a solubilidade do oxigênio dissolvido, afetando a vida aquática
2. **Alteração do ecossistema:** Espécies sensíveis à temperatura podem ser substituídas por espécies tolerantes, reduzindo a biodiversidade
3. **Proliferação de algas:** Temperaturas elevadas favorecem o crescimento de algas, podendo causar eutrofização

### Alternativas para Mitigação

1. **Torres de resfriamento:** Reduzem a temperatura da água antes do lançamento, mas consomem energia e água (evaporação)
2. **Lagos de resfriamento:** Armazenam água aquecida, permitindo resfriamento natural antes do lançamento
3. **Difusores de lançamento:** Misturam a água aquecida com a água do rio, reduzindo o gradiente térmico local
4. **Reuso da água:** Utilizam a água aquecida para aquicultura, irrigação ou aquecimento urbano
5. **Condensador a ar (ACC):** Eliminam o uso de água, mas reduzem a eficiência da usina (ver Caso 10.1)

In [ ]:
# ============================================================
# IMPACTO AMBIENTAL
# ============================================================

print('=' * 60)
print('IMPACTO AMBIENTAL DO LANÇAMENTO')
print('=' * 60)

# Comparação com vazões de rios
vazoes_rios = {
    'Rio pequeno': 3,
    'Rio médio': 75,
    'Rio grande': 750
}

print(f'\nVazão captada: {V_agua:.1f} m³/s')
print(f'\nComparação com vazões típicas:')
for nome, vazao in vazoes_rios.items():
    fracao = V_agua / vazao * 100
    print(f'  • {nome} ({vazao} m³/s): {fracao:.1f}% da vazão')

# Impacto na temperatura do rio
print(f'\n🌡️ Impacto térmico:')
print(f'  • Temperatura de entrada: {T_agua_e}°C')
print(f'  • Temperatura de saída: {T_agua_s}°C')
print(f'  • Elevação: {T_agua_s - T_agua_e}°C')

# Legislação ambiental
print(f'\n📋 Legislação (Resolução CONAMA 357/2005):')
print(f'  • Temperatura máxima no lançamento: 3°C acima da natural')
print(f'  • Zona de mistura autorizada: até 40°C')
print(f'  • Monitoramento contínuo obrigatório')

# Alternativas
print(f'\n💡 Alternativas de mitigação:')
print(f'  1. Torres de resfriamento (reduz ΔT em 10-15°C)')
print(f'  2. Lagos de resfriamento (armazenamento temporário)')
print(f'  3. Difusores de lançamento (mistura com rio)')
print(f'  4. Reuso da água (aquicultura, irrigação)')
print(f'  5. Condensador a ar (elimina uso de água)')

---

## 📝 Seção 11: Exercícios Resolvidos

### Exercício 1 (Graduação): Efeito do Verão na Área

**Enunciado:** Se a temperatura da água do rio subir para 28°C no verão (devido a uma estiagem), qual seria a nova área de troca necessária para manter a mesma carga térmica? Discuta as implicações para a eficiência da usina.

**Solução:** Já calculada na Seção 9:
- Nova LMTD: 16,5°C (vs. 22,6°C normal)
- Nova área: 5050 m² (vs. 3700 m² normal)
- Aumento: 36%

**Implicações:** Com área fixa de 3700 m², a usina não conseguirá condensar todo o vapor a 50°C. A temperatura de condensação aumentará, reduzindo a eficiência do ciclo Rankine e a potência elétrica gerada.

---

### Exercício 2 (Graduação): Número de Tubos

**Enunciado:** Calcule o número de tubos necessários, considerando tubos de cobre-níquel com $\varnothing$ externo 20 mm, $\varnothing$ interno 18 mm e comprimento 6 m.

**Solução:** Já calculada na Seção 6:
- Área por tubo: 0,377 m²
- Número de tubos: ~9800

---

### Exercício 3 (Graduação): Velocidade da Água

**Enunciado:** Verifique a velocidade da água nos tubos. Ela está dentro da faixa recomendada (1,5 a 2,5 m/s) para evitar incrustação e erosão?

**Solução:** Já calculada na Seção 7:
- Velocidade: 1,44 m/s
- Ligeiramente abaixo da faixa recomendada
- Risco de incrustação e biofilme

---

### Exercício 4 (Pós-Graduação): Incrustação após 1 Ano

**Enunciado:** Considere que após 1 ano de operação, a incrustação nos tubos elevou a resistência térmica para $R_f = 0,0002$ m²·K/W. Calcule o novo coeficiente global $U_{proj}$ e a nova temperatura de saída da água, mantendo a mesma área. A usina consegue manter os 100 MWe?

**Solução:**

O novo coeficiente global é:

$$\frac{1}{U_{proj}} = \frac{1}{U_{limpo}} + R_f = \frac{1}{1800} + 0,0002 = 0,000556 + 0,0002 = 0,000756$$

$$U_{proj} = \frac{1}{0,000756} \approx 1323 \text{ W/(m}^2\cdot\text{K)}$$

A carga térmica máxima com a área instalada é:

$$Q_{max} = U_{proj} \cdot A \cdot \Delta T_{lm} = 1323 \times 3700 \times 22,6 \approx 110,6 \text{ MW}$$

A potência elétrica máxima é:

$$W_{el,max} = \eta \cdot Q_{max} = 0,40 \times 110,6 \approx 44,2 \text{ MW}$$

**Conclusão:** A usina **não consegue** manter os 100 MWe. A potência cai para ~44 MW (redução de 56%). A limpeza dos tubos é urgente!

---

### Exercício 5 (Pós-Graduação): Impacto Ambiental

**Enunciado:** Compare o volume de água necessário (3,6 m³/s) com a vazão típica de um rio de médio porte. Discuta os impactos ambientais do lançamento de água aquecida e as alternativas para mitigação.

**Solução:** Já discutida na Seção 10:
- Para rio médio (50-100 m³/s): captação representa 3,6-7,2% da vazão
- Impactos: poluição térmica, alteração do ecossistema, proliferação de algas
- Alternativas: torres de resfriamento, lagos, difusores, reuso, ACC

In [ ]:
# ============================================================
# EXERCÍCIO 4 (PÓS): INCRUSTAÇÃO APÓS 1 ANO
# ============================================================

# Fator de incrustação
R_f = 0.0002  # m²·K/W

# Novo coeficiente global
U_proj = 1 / (1/U + R_f)

# Carga térmica máxima
Q_max = U_proj * A_comercial * LMTD / 1e6  # MW

# Potência elétrica máxima
W_el_max = eta * Q_max

# Redução percentual
reducao = (W_el - W_el_max) / W_el * 100

print('=' * 60)
print('EXERCÍCIO 4: INCRUSTAÇÃO APÓS 1 ANO')
print('=' * 60)
print(f'\nFator de incrustação: R_f = {R_f} m²·K/W')
print(f'\nCoeficiente global (limpo): U = {U} W/(m²·K)')
print(f'Coeficiente global (incrustado): U_proj = {U_proj:.0f} W/(m²·K)')
print(f'Redução: {(1 - U_proj/U)*100:.0f}%')
print(f'\nCarga térmica máxima: Q_max = {Q_max:.1f} MW')
print(f'Potência elétrica máxima: W_el,max = {W_el_max:.1f} MW')
print(f'Redução de potência: {reducao:.0f}%')

# Custo da perda de potência
tarifa = 300  # R$/MWh
horas_ano = 8000  # h/ano
perda_receita = (W_el - W_el_max) * tarifa * horas_ano / 1e6  # R$ milhões/ano

print(f'\n💰 Custo da perda de potência:')
print(f'  • Tarifa: R$ {tarifa}/MWh')
print(f'  • Operação: {horas_ano} h/ano')
print(f'  • Perda de receita: R$ {perda_receita:.1f} milhões/ano')
print(f'\n📋 Conclusão: Limpeza dos tubos é URGENTE!')
print(f'   Custo de limpeza (~R$ 2-5 milhões) é amplamente compensado')

---

## 💡 Seção 12: Discussão e Conclusões

### Principais Aprendizados

1. **Balanço energético:** A UTE rejeita 60% do calor fornecido pela caldeira no condensador, ilustrando a limitação da segunda lei da termodinâmica

2. **Dimensionamento:** O condensador requer uma área de troca de ~3700 m², equivalente a meio campo de futebol

3. **Sensibilidade:** A área necessária é altamente sensível à temperatura do rio, aumentando 36% no verão

4. **Impacto ambiental:** A captação de 3,6 m³/s e o lançamento de água aquecida exigem cuidados ambientais rigorosos

5. **Manutenção:** A incrustação reduz drasticamente o desempenho, exigindo limpeza periódica

### Trade-offs de Projeto

| Critério | Condensador a Água | Condensador a Ar (ACC) |
|----------|-------------------|------------------------|
| Eficiência | Alta (T_cond baixa) | Baixa (T_cond alta) |
| Consumo de água | Alto (3,6 m³/s) | Zero |
| Área de troca | Moderada (3700 m²) | Grande (114.000 m²) |
| Custo de investimento | Moderado | Alto |
| Impacto ambiental | Poluição térmica | Visual e sonoro |
| Confiabilidade | Alta | Média |

### Recomendações Práticas

1. **Projeto conservador:** Dimensionar para T_agua,e = 28°C (verão), não para a média anual

2. **Monitoramento contínuo:** Instalar sensores de temperatura, pressão diferencial e vazão para detectar incrustação precoce

3. **Cronograma de limpeza:** Planejar limpeza a cada 6-12 meses, dependendo da qualidade da água

4. **Estudo de impacto ambiental:** Elaborar EIA/RIMA para obtenção de licenças ambientais

5. **Alternativas:** Considerar ACC em regiões com escassez hídrica, mesmo com penalidade de eficiência

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 9](../notebooks/09_Trocadores_Calor.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 9.2**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>